In [ ]:
import os
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Display.SimpleGui import init_display
from OCC.Core.Quantity import Quantity_Color, Quantity_TOC_RGB

# ========== CONFIGURE PATHS ==========
INPUT_DIR = r'C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results_original'
# INPUT_DIR=r'C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\evaluation_results\test'
OUTPUT_DIR = r'C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_pngs_original'
# OUTPUT_DIR=r'C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\new_pngs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========== INIT OCC DISPLAY ==========
print("Initializing OCC display for batch PNG generation...")
display, _, _, _ = init_display()
display.View.SetImmediateUpdate(False)
params = display.View.RenderingParams()
params.NbMsaaSamples = 0
params.IsAntialiasingEnabled = False
display.View.SetBackgroundColor(Quantity_Color(1.0, 1.0, 1.0, Quantity_TOC_RGB))
try:
    display.View.SetWindowSize(1024, 768)
except Exception:
    pass

# ========== STEP TO PNG FUNCTION ==========
def step_to_iso_png(step_path, out_path):
    reader = STEPControl_Reader()
    if reader.ReadFile(step_path) != 1:
        print(f"[FAIL] Read failed: {step_path}")
        return False
    reader.TransferRoots()
    shape = reader.OneShape()
    if shape is None or shape.IsNull():
        print(f"[FAIL] Empty or invalid shape: {step_path}")
        return False
    try:
        display.EraseAll()
        display.DisplayShape(shape, update=False)
        display.View_Iso()
        display.FitAll()
        display.Repaint()
        display.View.Dump(out_path)
        print(f"[OK] {os.path.basename(out_path)}")
        return True
    except Exception as e:
        print(f"[FAIL] Render failed: {step_path} -> {e}")
        return False

# ========== BATCH GENERATE PNGS ==========
if __name__ == "__main__":
    files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".step", ".stp"))])
    print(f"Found {len(files)} STEP files.")
    for f in files:
        step_path = os.path.join(INPUT_DIR, f)
        png_name = os.path.splitext(f)[0] + ".png"
        png_path = os.path.join(OUTPUT_DIR, png_name)
        # if os.path.exists(png_path):
        #     print(f"[SKIP] Already exists: {png_name}")
        #     continue
        step_to_iso_png(step_path, png_path)
    print("Batch PNG generation complete.")


Initializing OCC display for batch PNG generation...
INFO:OCC.Display.backend:The qt-pyqt5 backend is already loaded...``load_backend`` can only be called once per session
qt-pyqt5 backend - Qt version 5.15.2
Found 3544 STEP files.
[OK] 00000134_vec.png
[OK] 00000392_vec.png
[OK] 00000633_vec.png
[OK] 00000659_vec.png
[OK] 00000695_vec.png
[OK] 00000915_vec.png
[OK] 00001220_vec.png
[OK] 00001402_vec.png
[OK] 00001537_vec.png
[OK] 00001615_vec.png
[OK] 00001734_vec.png
[OK] 00002919_vec.png
[OK] 00003666_vec.png
[OK] 00003775_vec.png
[OK] 00004282_vec.png
[OK] 00004981_vec.png
[OK] 00005085_vec.png
[OK] 00005191_vec.png
[OK] 00005413_vec.png
[OK] 00005414_vec.png
[OK] 00005418_vec.png
[OK] 00005422_vec.png
[OK] 00005707_vec.png
[OK] 00005791_vec.png
[OK] 00005793_vec.png
[OK] 00005837_vec.png
[OK] 00006222_vec.png
[OK] 00006848_vec.png
[OK] 00006919_vec.png
[OK] 00007193_vec.png
[OK] 00007553_vec.png
[OK] 00007883_vec.png
[OK] 00008056_vec.png
[OK] 00008065_vec.png
[OK] 00008167_vec.pn

In [7]:
import os
import h5py
import numpy as np
from flask import Flask, render_template_string, send_from_directory

# ================= PATHS =================
# INPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\evaluation_results\test"
INPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results"
# OUTPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\new_pngs"
OUTPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\new_pngs_new_views"
# SVG_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw"
SVG_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_vaish"
CAD_VEC_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\cad_vec"
H5_DIR = INPUT_DIR  # Same as INPUT_DIR for predicted h5 files

# ================= METRIC CONSTANTS =================
CAD_EOS_IDX = 2
GENERATED_DATASET_NAME = "out_vec"
TRUTH_DATASET_NAME = "vec"
TOL = 3          # parameter tolerance
PAD_PARAM = -1   # padded argument marker

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================= FLASK APP =================
app = Flask(__name__)

# ================= METRIC FUNCTIONS =================
def load_pair(gen_path, truth_path):
    try:
        with h5py.File(truth_path, "r") as f:
            truth = f[TRUTH_DATASET_NAME][:]
        with h5py.File(gen_path, "r") as f:
            gen = f[GENERATED_DATASET_NAME][:]
    except Exception as e:
        print(f"[ERROR] Failed to load h5 pair: {gen_path}, {truth_path} -> {e}")
        return None
    T = min(len(gen), len(truth))
    gen = gen[:T]
    truth = truth[:T]
    C = min(gen.shape[1], truth.shape[1])
    gen = gen[:, :C]
    truth = truth[:, :C]
    return gen, truth

def compute_ACCcmd(pred_cmd, gt_cmd):
    valid = (gt_cmd != CAD_EOS_IDX)
    Nc = valid.sum()
    if Nc == 0:
        return 0.0
    correct = (pred_cmd == gt_cmd) & valid
    return 100.0 * correct.sum() / Nc

def compute_ACCparam(pred_args, gt_args, pred_cmd, gt_cmd, eta=3):
    valid_cmd = (gt_cmd != CAD_EOS_IDX)
    correct_cmd = (pred_cmd == gt_cmd) & valid_cmd
    valid_param = (gt_args != PAD_PARAM)
    mask = valid_param & correct_cmd[:, None]
    K = mask.sum()
    if K == 0:
        return 0.0
    abs_diff = np.abs(gt_args - pred_args)
    correct_param = (abs_diff <= eta)
    return 100.0 * (correct_param & mask).sum() / K

def compute_metrics_for_file(gen_h5_path, truth_h5_path):
    pair = load_pair(gen_h5_path, truth_h5_path)
    if pair is None:
        print(f"[WARN] Could not load pair for metrics: {gen_h5_path}, {truth_h5_path}")
        return None, None
    gen, truth = pair
    pred_cmd = gen[:, 0]
    gt_cmd = truth[:, 0]
    pred_args = gen[:, 1:]
    gt_args = truth[:, 1:]
    ACCcmd = compute_ACCcmd(pred_cmd, gt_cmd)
    ACCparam = compute_ACCparam(pred_args, gt_args, pred_cmd, gt_cmd, eta=TOL)
    return ACCcmd, ACCparam

def find_ground_truth_h5(step_name):
    base_id = step_name.replace("_vec.step", "").replace(".step", "")
    try:
        n = int(base_id)
    except Exception:
        print(f"[WARN] Could not parse base_id from step name: {step_name}")
        return None
    bucket = f"{n // 10000:04d}"
    h5_path = os.path.join(CAD_VEC_ROOT, bucket, f"{base_id}.h5")
    if not os.path.exists(h5_path):
        print(f"[WARN] Ground truth h5 not found: {h5_path}")
    return h5_path if os.path.exists(h5_path) else None

def read_h5_content(h5_path, dataset_name):
    try:
        with h5py.File(h5_path, "r") as f:
            data = f[dataset_name][:]
        cmd_names = {0: 'Line', 1: 'Arc', 2: 'Circle', 3: 'EOS', 4: 'SOL', 5: 'Ext'}
        formatted = []
        for i, row in enumerate(data):
            cmd_type = int(row[0])
            params = row[1:].astype(int)
            cmd_name = cmd_names.get(cmd_type, f'Unknown({cmd_type})')
            param_strs = []
            if cmd_type == 0:
                if params[0] != -1: param_strs.append(f"x:{params[0]}")
                if params[1] != -1: param_strs.append(f"y:{params[1]}")
            elif cmd_type == 1:
                if params[0] != -1: param_strs.append(f"x:{params[0]}")
                if params[1] != -1: param_strs.append(f"y:{params[1]}")
                if params[2] != -1: param_strs.append(f"α:{params[2]}")
                if params[3] != -1: param_strs.append(f"f:{params[3]}")
            elif cmd_type == 2:
                if params[0] != -1: param_strs.append(f"x:{params[0]}")
                if params[1] != -1: param_strs.append(f"y:{params[1]}")
                if params[4] != -1: param_strs.append(f"r:{params[4]}")
            elif cmd_type == 5:
                if params[5] != -1: param_strs.append(f"θ:{params[5]}")
                if params[6] != -1: param_strs.append(f"φ:{params[6]}")
                if params[7] != -1: param_strs.append(f"γ:{params[7]}")
                if params[8] != -1: param_strs.append(f"px:{params[8]}")
                if params[9] != -1: param_strs.append(f"py:{params[9]}")
                if params[10] != -1: param_strs.append(f"pz:{params[10]}")
                if params[11] != -1: param_strs.append(f"s:{params[11]}")
                if params[12] != -1: param_strs.append(f"e1:{params[12]}")
                if params[13] != -1: param_strs.append(f"e2:{params[13]}")
                if params[14] != -1: param_strs.append(f"b:{params[14]}")
                if params[15] != -1: param_strs.append(f"u:{params[15]}")
            if param_strs:
                formatted.append(f"⟨{cmd_name}⟩_{i}: ({', '.join(param_strs)})")
            else:
                formatted.append(f"⟨{cmd_name}⟩_{i}: ∅")
        return '\n'.join(formatted)
    except Exception as e:
        print(f"[ERROR] Failed to read h5 content: {h5_path}, {dataset_name} -> {e}")
        return f"Error reading file: {e}"

# ============== PROCESS ALL STEPS ==============
def process_all_steps():
    items = []
    for f in sorted(os.listdir(INPUT_DIR)):
        if not f.lower().endswith((".step", ".stp")):
            continue
        step_path = os.path.join(INPUT_DIR, f)
        png_name = os.path.splitext(f)[0] + ".png"
        png_path = os.path.join(OUTPUT_DIR, png_name)
        # Only use already-generated PNGs
        if not os.path.exists(png_path):
            print(f"[SKIP] PNG not found for: {f}")
            continue
        svg = find_original_svg(f)
        # --- FIX: base_id should keep _vec if present ---
        base_id = f.replace(".step", "").replace(".stp", "")
        gen_h5_path = os.path.join(H5_DIR, base_id + ".h5")
        if not os.path.exists(gen_h5_path):
            # Try with _vec.h5 if not found
            gen_h5_path_vec = os.path.join(H5_DIR, base_id + "_vec.h5")
            if os.path.exists(gen_h5_path_vec):
                print(f"[FOUND] Generated h5 (with _vec) for: {f} | Path: {gen_h5_path_vec}")
                gen_h5_path = gen_h5_path_vec
            else:
                print(f"[SKIP] Generated h5 missing for: {f} | Checked path: {gen_h5_path} and {gen_h5_path_vec} | Exists: False")
        else:
            print(f"[FOUND] Generated h5 for: {f} | Path: {gen_h5_path}")
        truth_h5_path = find_ground_truth_h5(f)
        acc_cmd, acc_param = None, None
        gen_h5_content, truth_h5_content = None, None
        if truth_h5_path and os.path.exists(gen_h5_path):
            acc_cmd, acc_param = compute_metrics_for_file(gen_h5_path, truth_h5_path)
            gen_h5_content = read_h5_content(gen_h5_path, GENERATED_DATASET_NAME)
            truth_h5_content = read_h5_content(truth_h5_path, TRUTH_DATASET_NAME)
        else:
            if not truth_h5_path:
                print(f"[SKIP] Truth h5 missing for: {f}")
            if not os.path.exists(gen_h5_path):
                print(f"[SKIP] Generated h5 missing for: {f}")
        items.append({
            "step": f,
            "png": png_name,
            "svg": svg,
            "acc_cmd": acc_cmd,
            "acc_param": acc_param,
            "gen_h5": gen_h5_content,
            "truth_h5": truth_h5_content
        })
    print(f"[SUMMARY] Total processed: {len(items)}")
    return items

def find_original_svg(step_name):
    base_id = step_name.replace("_vec.step", "").replace(".step", "").split(".")[0]
    try:
        n = int(base_id)
    except Exception:
        print(f"[WARN] Could not parse base_id for SVG: {step_name}")
        return None
    bucket = f"{n // 10000:04d}"
    svg_path = os.path.join(SVG_ROOT, bucket, base_id, f"{base_id}_FrontTopRight.svg")
    if not os.path.exists(svg_path):
        print(f"[SKIP] SVG not found: {svg_path}")
    return svg_path if os.path.exists(svg_path) else None

# ============== FLASK ROUTES ==============
@app.route('/')
def index():
    items = process_all_steps()
    html_template = """<!DOCTYPE html>
<html>
<head>
<meta charset=\"utf-8\">
<title>STEP Isometric Comparison - Live View</title>
<style>
body { font-family: Arial, sans-serif; background: #f2f2f2; margin: 0; padding: 20px 0; }
.container { max-width: 1200px; margin: auto; }
h1 { text-align: center; margin: 8px 0 20px 0; }
.info { text-align: center; color: #666; margin-bottom: 20px; font-size: 14px; }
.grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; align-items: start; }
.stepname { grid-column: span 2; font-weight: bold; padding: 6px 0; }
.card { background: white; border-radius: 6px; padding: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.12); text-align: center; min-height: 260px; max-height: 500px; display: flex; flex-direction: column; justify-content: flex-start; align-items: center; overflow: hidden; }
.card .label { font-size: 12px; color: #666; margin-bottom: 6px; }
.metrics { font-size: 11px; color: #0066cc; margin: 6px 0; font-weight: 600; }
.card img { width: 100%; height: auto; object-fit: contain; border-radius: 4px; background: #fff; }
.placeholder { color: #999; font-size: 13px; margin-top: 40px; }
.small { font-size: 12px; color: #444; margin-top: 6px; word-break: break-all; }
.h5-info { font-size: 9px; color: #333; margin-top: 8px; padding: 6px; background: #f8f8f8; border-radius: 3px; font-family: 'Courier New', monospace; text-align: left; max-height: 150px; overflow-y: auto; white-space: pre-wrap; word-break: break-all; }
</style>
</head>
<body>
<div class=\"container\">
<h1>🔴 LIVE: STEP Isometric Comparison</h1>
<div class=\"info\">Flask Server Running | Total Files: {{ total_count }}</div>
<div class=\"grid\">
{% set n = items|length %}
{% for i in range(0, n, 2) %}
    {% set a = items[i] %}
    {% set b = items[i+1] if (i+1) < n else None %}
    <div class=\"stepname\">{{ a.step }}
    {% if a.acc_cmd is not none %}
        <span style=\"color:#0066cc;\">[ACCcmd: {{ "%.2f"|format(a.acc_cmd) }}% | ACCparam: {{ "%.2f"|format(a.acc_param) }}%]</span>
    {% endif %}
    </div>
    {% if b %}
    <div class=\"stepname\">{{ b.step }}
    {% if b.acc_cmd is not none %}
        <span style=\"color:#0066cc;\">[ACCcmd: {{ "%.2f"|format(b.acc_cmd) }}% | ACCparam: {{ "%.2f"|format(b.acc_param) }}%]</span>
    {% endif %}
    </div>
    {% else %}
    <div class=\"stepname\"></div>
    {% endif %}
    <div class=\"card\">
        <div class=\"label\">Generated</div>
        <img src="/png/{{ a.png }}" alt="generated">
        <div class=\"small\">{{ a.png }}</div>
        {% if a.gen_h5 %}
        <div class=\"h5-info\">{{ a.gen_h5 }}</div>
        {% endif %}
    </div>
    <div class=\"card\">
        <div class=\"label\">Original</div>
        {% if a.svg %}
            <img src="/svg/{{ a.svg }}" alt="original">
            <div class=\"small\">Original SVG</div>
        {% else %}
            <div class=\"placeholder\">Not available</div>
        {% endif %}
        {% if a.truth_h5 %}
        <div class=\"h5-info\">{{ a.truth_h5 }}</div>
        {% endif %}
    </div>
    {% if b %}
    <div class=\"card\">
        <div class=\"label\">Generated</div>
        <img src="/png/{{ b.png }}" alt="generated">
        <div class=\"small\">{{ b.png }}</div>
        {% if b.gen_h5 %}
        <div class=\"h5-info\">{{ b.gen_h5 }}</div>
        {% endif %}
    </div>
    {% else %}
    <div class=\"card\"><div class=\"placeholder\">--</div></div>
    {% endif %}
    {% if b %}
    <div class=\"card\">
        <div class=\"label\">Original</div>
        {% if b.svg %}
            <img src="/svg/{{ b.svg }}" alt="original">
            <div class=\"small\">Original SVG</div>
        {% else %}
            <div class=\"placeholder\">Not available</div>
        {% endif %}
        {% if b.truth_h5 %}
        <div class=\"h5-info\">{{ b.truth_h5 }}</div>
        {% endif %}
    </div>
    {% else %}
    <div class=\"card\"><div class=\"placeholder\">--</div></div>
    {% endif %}
{% endfor %}
</div>
</div>
</body>
</html>
"""
    return render_template_string(html_template, items=items, total_count=len(items))

@app.route('/png/<filename>')
def serve_png(filename):
    return send_from_directory(OUTPUT_DIR, filename)

@app.route('/svg/<path:path>')
def serve_svg(path):
    directory = os.path.dirname(path)
    filename = os.path.basename(path)
    return send_from_directory(directory, filename)

def run_flask_in_notebook():
    print("=" * 60)
    print("🚀 Starting Flask server...")
    print("=" * 60)
    print(f"📂 Input DIR: {INPUT_DIR}")
    print(f"📂 Output DIR: {OUTPUT_DIR}")
    print(f"📂 SVG ROOT: {SVG_ROOT}")
    print(f"📂 CAD VEC ROOT: {CAD_VEC_ROOT}")
    print("=" * 60)
    print("🌐 Server will start at: http://127.0.0.1:5000")
    print("=" * 60)
    app.run(debug=True, host='0.0.0.0', port=5000, use_reloader=False)

In [8]:
# To run the Flask app in a notebook, call this function in a cell below:
run_flask_in_notebook()

🚀 Starting Flask server...
📂 Input DIR: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results
📂 Output DIR: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\new_pngs_new_views
📂 SVG ROOT: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_vaish
📂 CAD VEC ROOT: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\cad_vec
🌐 Server will start at: http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.10.20.84:5000
Press CTRL+C to quit


[FOUND] Generated h5 for: 00000134_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000134_vec.h5
[FOUND] Generated h5 for: 00000392_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000392_vec.h5
[FOUND] Generated h5 for: 00000633_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000633_vec.h5
[FOUND] Generated h5 for: 00000659_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000659_vec.h5
[FOUND] Generated h5 for: 00000695_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000695_vec.h5
[FOUND] Generated h5 for: 00000915_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000915_vec.h5
[FOUND] Generated h5 for: 00001220_vec.step | Path: C:\Users\LEG

127.0.0.1 - - [22/Dec/2025 13:57:47] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /png/00000134_vec.png HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00000134/00000134_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /png/00000392_vec.png HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /png/00000633_vec.png HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00000392/00000392_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00000633/00000633_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /png/00000659_vec.png HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 13:57:47] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/

[FOUND] Generated h5 for: 00000134_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000134_vec.h5
[FOUND] Generated h5 for: 00000392_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000392_vec.h5
[FOUND] Generated h5 for: 00000633_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000633_vec.h5
[FOUND] Generated h5 for: 00000659_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000659_vec.h5
[FOUND] Generated h5 for: 00000695_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000695_vec.h5
[FOUND] Generated h5 for: 00000915_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000915_vec.h5
[FOUND] Generated h5 for: 00001220_vec.step | Path: C:\Users\LEG

...

127.0.0.1 - - [22/Dec/2025 14:05:59] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00000392/00000392_FrontTopRight.svg HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:05:59] "GET /png/00000633_vec.png HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:06:00] "GET /png/00001220_vec.png HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:06:00] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00001734/00001734_FrontTopRight.svg HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:06:00] "GET /png/00004981_vec.png HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:06:00] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00004981/00004981_FrontTopRight.svg HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:06:00] "GET /png/00002919_vec.png HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 14:06:00] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00001220/00001220_FrontTopRight.svg HTTP/1.1" 200 -
127.0.0.

[FOUND] Generated h5 for: 00000134_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000134_vec.h5
[FOUND] Generated h5 for: 00000392_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000392_vec.h5
[FOUND] Generated h5 for: 00000633_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000633_vec.h5
[FOUND] Generated h5 for: 00000659_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000659_vec.h5
[FOUND] Generated h5 for: 00000695_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000695_vec.h5
[FOUND] Generated h5 for: 00000915_vec.step | Path: C:\Users\LEGION\Desktop\cad_project\Drawing2CAD\proj_log\epoch_100_shivank\test_results\00000915_vec.h5
[FOUND] Generated h5 for: 00001220_vec.step | Path: C:\Users\LEG

127.0.0.1 - - [22/Dec/2025 14:08:36] "GET / HTTP/1.1" 200 -


[SUMMARY] Total processed: 693


127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00000633/00000633_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00000392/00000392_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00002919/00002919_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /png/00001402_vec.png HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /png/00001220_vec.png HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0000/00009049/00009049_FrontTopRight.svg HTTP/1.1" 304 -
127.0.0.1 - - [22/Dec/2025 14:08:37] "GET /svg/C:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_vaish/0001/00013674/00013674_FrontTopRight.svg HTTP/1.1" 304 -


In [1]:
import os
import random
import h5py
import numpy as np
import sys
sys.path.append(r"C:\Users\LEGION\Desktop\cad_project\DeepCAD")
from DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Display.SimpleGui import init_display

# ================== CONFIG ==================
CAD_VEC_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\cad_vec"
STEP_OUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_Steps"
PNG_OUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_PNGs"
H5_SHAPE_KEY = "vec"      # <-- change if needed
PNG_SIZE = (800, 800)
NUM_SAMPLES = 100
os.makedirs(STEP_OUT_DIR, exist_ok=True)
os.makedirs(PNG_OUT_DIR, exist_ok=True)

# ================== H5 TO STEP ==================
def h5_to_step_vec2cad(h5_path, step_path):
    with h5py.File(h5_path, "r") as f:
        if H5_SHAPE_KEY not in f:
            raise ValueError(f"Key '{H5_SHAPE_KEY}' not found in {h5_path}")
        vec = f[H5_SHAPE_KEY][:].astype(np.float64)
        shape = vec2CADsolid(vec)
    write_step_file(shape, step_path)
    return shape

# ================== STEP TO PNG ==================
def step_to_iso_png(step_path, png_path):
    display, _, _, _ = init_display(size=PNG_SIZE)
    display.View.SetImmediateUpdate(False)
    params = display.View.RenderingParams()
    params.NbMsaaSamples = 0
    params.IsAntialiasingEnabled = False
    reader = STEPControl_Reader()
    if reader.ReadFile(step_path) != 1:
        print(f"[FAIL] Read failed: {step_path}")
        return False
    reader.TransferRoots()
    shape = reader.OneShape()
    if shape is None or shape.IsNull():
        print(f"[FAIL] Empty or invalid shape: {step_path}")
        return False
    try:
        display.EraseAll()
        display.DisplayShape(shape, update=False)
        display.View_Iso()
        display.FitAll()
        display.Repaint()
        display.View.Dump(png_path)
        display.EraseAll()
        display.Close()
        print(f"[OK] {os.path.basename(png_path)}")
        return True
    except Exception as e:
        print(f"[FAIL] Render failed: {step_path} -> {e}")
        return False

# ================== RANDOM SAMPLE & CONVERT ==================
def get_all_h5_files(cad_vec_root):
    h5_files = []
    for root, _, files in os.walk(cad_vec_root):
        for f in files:
            if f.endswith('.h5'):
                h5_files.append(os.path.join(root, f))
    return h5_files

all_h5_files = get_all_h5_files(CAD_VEC_ROOT)
print(f"Found {len(all_h5_files)} h5 files.")
if len(all_h5_files) < NUM_SAMPLES:
    print(f"⚠ Only {len(all_h5_files)} available, using all.")
    sample_h5_files = all_h5_files
else:
    sample_h5_files = random.sample(all_h5_files, NUM_SAMPLES)

failed = []
for h5_path in sample_h5_files:
    base = os.path.splitext(os.path.basename(h5_path))[0]
    step_path = os.path.join(STEP_OUT_DIR, base + ".step")
    png_path = os.path.join(PNG_OUT_DIR, base + ".png")
    try:
        shape = h5_to_step_vec2cad(h5_path, step_path)
        ok = step_to_iso_png(step_path, png_path)
        if not ok:
            failed.append(h5_path)
    except Exception as e:
        print(f"❌ {h5_path} | {e}")
        failed.append(h5_path)

print("\n========== SUMMARY ==========")
print(f"Total failed files: {len(failed)}")
if failed:
    print("\nExclude these .h5 files from training:")
    for f in failed:
        print(f" - {f}")


Found 157591 h5 files.
qt-pyqt5 backend - Qt version 5.15.2
[FAIL] Render failed: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_Steps\00038386.step -> 'Viewer3d' object has no attribute 'Close'
INFO:OCC.Display.backend:The qt-pyqt5 backend is already loaded...``load_backend`` can only be called once per session
qt-pyqt5 backend - Qt version 5.15.2
[FAIL] Render failed: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_Steps\00352942.step -> 'Viewer3d' object has no attribute 'Close'
INFO:OCC.Display.backend:The qt-pyqt5 backend is already loaded...``load_backend`` can only be called once per session
qt-pyqt5 backend - Qt version 5.15.2
[FAIL] Render failed: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_Steps\00710694.step -> 'Viewer3d' object has no attribute 'Close'
INFO:OCC.Display.backend:The qt-pyqt5 backend is already loaded...``load_backend`` can only be called once per session
qt-pyqt5 backend - Qt version 5.15.2
[FAIL] Render failed: C:\Use

In [2]:
import os
import random
from pathlib import Path

# ================= CONFIG =================
GT_SVG_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw"
PRED_PNG_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_PNGs"
OUTPUT_HTML = "svg_vs_pred_comparison.html"
NUM_SAMPLES = 100
IMG_WIDTH = 300

# ==========================================
def find_svg_path_from_png_name(gt_svg_root, png_name):
    base = os.path.splitext(png_name)[0]
    bucket = base[:4]
    svg_path = os.path.join(gt_svg_root, bucket, base, f"{base}_FrontTopRight.svg")
    return svg_path if os.path.exists(svg_path) else None

def generate_svg_vs_pred_html(gt_svg_root, pred_png_dir, output_html, num_samples):
    pred_files = [f for f in os.listdir(pred_png_dir) if f.lower().endswith('.png')]
    print(f"Total predicted PNGs: {len(pred_files)}")
    pairs = []
    for png_name in pred_files:
        svg_path = find_svg_path_from_png_name(gt_svg_root, png_name)
        if svg_path:
            # Convert Windows paths to URI format for HTML (forward slashes, no backslashes)
            svg_uri = 'file:///' + svg_path.replace('\\', '/')
            png_uri = 'file:///' + os.path.join(pred_png_dir, png_name).replace('\\', '/')
            pairs.append((svg_uri, png_uri, png_name))
    print(f"Total matching SVG/PNG pairs: {len(pairs)}")
    if len(pairs) == 0:
        raise RuntimeError("No matching SVG/PNG pairs found")
    if len(pairs) < num_samples:
        print(f"⚠ Only {len(pairs)} available, using all.")
        samples = pairs
    else:
        samples = random.sample(pairs, num_samples)
    rows = []
    for svg_uri, png_uri, fname in samples:
        row = (
            "<div class='row'>"
            f"<div class='cell'><img src='{svg_uri}' width='{IMG_WIDTH}'><p>GT SVG</p></div>"
            f"<div class='cell'><img src='{png_uri}' width='{IMG_WIDTH}'><p>PRED PNG</p></div>"
            f"<div class='fname'>{fname}</div>"
            "</div>"
        )
        rows.append(row)
    html = (
        "<!DOCTYPE html>"
        "<html>"
        "<head>"
        "<meta charset='UTF-8'>"
        "<title>SVG vs Prediction Comparison</title>"
        "<style>"
        "body { font-family: Arial, sans-serif; background: #111; color: #eee; }"
        ".row { display: flex; align-items: center; margin-bottom: 20px; border-bottom: 1px solid #333; padding-bottom: 10px; }"
        ".cell { margin-right: 20px; text-align: center; }"
        "img { border: 1px solid #444; background: white; }"
        ".fname { color: #aaa; font-size: 14px; }"
        "</style>"
        "</head>"
        "<body>"
        f"<h1>SVG vs Prediction (Random {len(samples)})</h1>"
        + ''.join(rows) +
        "</body>"
        "</html>"
    )
    with open(output_html, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ HTML saved to: {output_html}")

# Run the function
generate_svg_vs_pred_html(GT_SVG_ROOT, PRED_PNG_DIR, OUTPUT_HTML, NUM_SAMPLES)


Total predicted PNGs: 99
Total matching SVG/PNG pairs: 99
⚠ Only 99 available, using all.
✅ HTML saved to: svg_vs_pred_comparison.html


In [1]:
import os
import random
from pathlib import Path

# ================= CONFIG =================
GT_SVG_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw"
PRED_PNG_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_PNGs"
NPY_SVG_ROOT = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs"
OUTPUT_HTML = "svg_vs_pred_comparison2.html"
NUM_SAMPLES = 100
IMG_WIDTH = 280

# ==========================================
def find_svg_path_from_png_name(gt_svg_root, png_name):
    base = os.path.splitext(png_name)[0]
    bucket = base[:4]
    svg_path = os.path.join(gt_svg_root, bucket, base, f"{base}_FrontTopRight.svg")
    return svg_path if os.path.exists(svg_path) else None

def find_npy_svg_path_from_png_name(npy_svg_root, png_name):
    base = os.path.splitext(png_name)[0]
    npy_svg_path = os.path.join(npy_svg_root, base, f"{base}_isometric.svg")
    return npy_svg_path if os.path.exists(npy_svg_path) else None

def generate_3way_comparison_html(gt_svg_root, pred_png_dir, npy_svg_root, output_html, num_samples):
    pred_files = [f for f in os.listdir(pred_png_dir) if f.lower().endswith('.png')]
    print(f"Total predicted PNGs: {len(pred_files)}")
    triples = []
    for png_name in pred_files:
        svg_path = find_svg_path_from_png_name(gt_svg_root, png_name)
        npy_svg_path = find_npy_svg_path_from_png_name(npy_svg_root, png_name)
        if svg_path or npy_svg_path:  # Include if at least one GT exists
            svg_uri = 'file:///' + svg_path.replace('\\', '/') if svg_path else None
            png_uri = 'file:///' + os.path.join(pred_png_dir, png_name).replace('\\', '/')
            npy_svg_uri = 'file:///' + npy_svg_path.replace('\\', '/') if npy_svg_path else None
            triples.append((svg_uri, png_uri, npy_svg_uri, png_name))
    print(f"Total matching triples: {len(triples)}")
    if len(triples) == 0:
        raise RuntimeError("No matching files found")
    if len(triples) < num_samples:
        print(f"⚠ Only {len(triples)} available, using all.")
        samples = triples
    else:
        samples = random.sample(triples, num_samples)
    rows = []
    for svg_uri, png_uri, npy_svg_uri, fname in samples:
        row = "<div class='row'>"
        if svg_uri:
            row += f"<div class='cell'><img src='{svg_uri}' width='{IMG_WIDTH}'><p>GT SVG (Raw)</p></div>"
        else:
            row += f"<div class='cell'><div class='placeholder'>GT SVG N/A</div></div>"
        row += f"<div class='cell'><img src='{png_uri}' width='{IMG_WIDTH}'><p>PRED PNG</p></div>"
        if npy_svg_uri:
            row += f"<div class='cell'><img src='{npy_svg_uri}' width='{IMG_WIDTH}'><p>Npy SVG</p></div>"
        else:
            row += f"<div class='cell'><div class='placeholder'>Npy SVG N/A</div></div>"
        row += f"<div class='fname'>{fname}</div></div>"
        rows.append(row)
    html = (
        "<!DOCTYPE html>"
        "<html>"
        "<head>"
        "<meta charset='UTF-8'>"
        "<title>3-Way Comparison: GT SVG vs PRED PNG vs Npy SVG</title>"
        "<style>"
        "body { font-family: Arial, sans-serif; background: #111; color: #eee; padding: 20px; }"
        ".row { display: flex; align-items: center; margin-bottom: 20px; border-bottom: 1px solid #333; padding-bottom: 10px; }"
        ".cell { margin-right: 20px; text-align: center; }"
        "img { border: 1px solid #444; background: white; }"
        ".fname { color: #aaa; font-size: 14px; margin-left: auto; }"
        ".placeholder { color: #666; font-size: 12px; padding: 40px 20px; }"
        "</style>"
        "</head>"
        "<body>"
        f"<h1>3-Way Comparison (Random {len(samples)} samples)</h1>"
        + ''.join(rows) +
        "</body>"
        "</html>"
    )
    with open(output_html, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ HTML saved to: {output_html}")

# Run the function
generate_3way_comparison_html(GT_SVG_ROOT, PRED_PNG_DIR, NPY_SVG_ROOT, OUTPUT_HTML, NUM_SAMPLES)


Total predicted PNGs: 99
Total matching triples: 99
⚠ Only 99 available, using all.
✅ HTML saved to: svg_vs_pred_comparison2.html


In [ ]:
import subprocess
import os

# ==================== STEP TO SVG CONVERSION ====================
# This cell runs the FreeCAD script to convert STEP files to SVG drawings
# Requirements: FreeCAD must be installed

FREECAD_EXE = r"C:\Program Files\FreeCAD 1.0\bin\FreeCAD.exe"
SCRIPT_PATH = r"C:\Users\LEGION\Desktop\cad_project\step_to_svg_freecad.py"

def run_freecad_script():
    """Run the FreeCAD script to convert STEP files to SVG."""
    if not os.path.exists(FREECAD_EXE):
        print("❌ FreeCAD not found at:", FREECAD_EXE)
        print("Please update FREECAD_EXE path in the cell")
        return
    
    if not os.path.exists(SCRIPT_PATH):
        print("❌ Script not found at:", SCRIPT_PATH)
        return
    
    print("🚀 Starting FreeCAD script...")
    print("=" * 60)
    
    # Run FreeCAD in console mode with the script
    cmd = [FREECAD_EXE, "-c", SCRIPT_PATH]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace'
    )
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    if result.returncode == 0:
        print("=" * 60)
        print("✅ FreeCAD script completed successfully!")
    else:
        print("=" * 60)
        print(f"❌ FreeCAD script failed with return code: {result.returncode}")

# Uncomment the line below to run the conversion
# run_freecad_script()
